# Fake News Detection - Data Exploration
This notebook explores the fake news dataset, demonstrates text preprocessing, and visualizes model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.abspath('..'))
%matplotlib inline

## 1. Data Loading and Exploration

In [ ]:
df = pd.read_csv('../datasets/sample_data.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
df.info()
print('\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nPercentage of fake news: {df["label"].mean() * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Label Distribution')
axes[0].set_xticklabels(['Real (0)', 'Fake (1)'])

df['title_length'] = df['title'].apply(len)
df['text_length'] = df['text'].apply(len)

df.boxplot(column='text_length', by='label', ax=axes[1])
axes[1].set_title('Text Length by Label')
axes[1].set_xticklabels(['Real', 'Fake'])
plt.tight_layout()
plt.show()

## 2. Text Preprocessing Demo

In [ ]:
from backend.utils.preprocessing import preprocess_text, ensure_nltk_data

ensure_nltk_data()

sample_text = df['text'].iloc[0]
print('Original:')
print(sample_text[:200])
print('\nPreprocessed:')
print(preprocess_text(sample_text)[:200])

In [ ]:
processed_examples = []
for i in range(5):
    row = df.iloc[i]
    processed_examples.append({
        'title': row['title'],
        'label': 'Fake' if row['label'] else 'Real',
        'processed': preprocess_text(row['text'])[:100]
    })

pd.DataFrame(processed_examples)

## 3. Model Training and Visualization

In [ ]:
from backend.training.train import main as train_main

pipeline, results, label_encoder = train_main()

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('accuracy', ascending=False)
print('Model Performance Comparison:')
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
results_df[['accuracy', 'precision', 'recall', 'f1_score']].plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
ax.legend(loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import joblib
from sklearn.metrics import confusion_matrix, classification_report

df_test = pd.read_csv('../datasets/sample_data.csv')
y_true = df_test['label']
y_pred = pipeline.predict(df_test['text'])

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

In [ ]:
print('Notebook complete. The pipeline is ready for inference.')